## 🗺️ Lab Map: The Bike Demand Forecasting Flow

Below is the end-to-end journey for this lab. Each step represents a module in this workshop, moving from raw data to a monitored production model:

**Explore** $\xrightarrow{}$ **Prepare** $\xrightarrow{}$ **Train** $\xrightarrow{}$ **Register** $\xrightarrow{}$ **Serve** $\xrightarrow{}$ **Monitor**

--- 
*Current Module: **Register** (Module 04)*

# 🚀 Module 4: Review the Experiments & Select the Best Model

### ✅ You're done when...
- The best run is identified using the logged RMSE and R² metrics.
- The selected model is successfully registered in the MLflow Model Registry.
- The model is assigned a version number (e.g., v1).

In [ ]:
# Install requirements
!pip install -r ../requirements.txt

## 📦 Import Required Libraries

Before we proceed with training and tracking our machine learning model, we need to import the necessary libraries.


In [ ]:
# Import necessary modules
import os
import random

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

## ✅ Set Necessary Variables

In [ ]:
# TODO: Fill in the required variables

# Disable TLS verification for self-signed cert
os.environ['MLFLOW_TRACKING_INSECURE_TLS'] = 'true'

# Variables for model tracking
MLFLOW_REMOTE_TRACKING_SERVER = ___  # Hint: the MLflow tracking URI from your environment

# Unique name for the MLflow experiment
YOUR_FIRSTNAME = ___  # Hint: your first name (e.g., "john")


## 📋 Retrieve and Review Experiment Runs

We query the MLflow tracking server to retrieve all runs associated with the "bike_sharing_model" experiment:

- Use MlflowClient to fetch all runs.

- Extract relevant details such as Run ID, Run Name, RMSE, R², and Start Time.

- Display the runs in a Pandas DataFrame for easier inspection and comparison.

The runs are sorted by start date, allowing us to review recent experiments and identify the best-performing model based on evaluation metrics.

In [ ]:
MLFLOW_TRACKING_URI = MLFLOW_REMOTE_TRACKING_SERVER
PARTICIPANT_FIRSTNAME = YOUR_FIRSTNAME 

os.environ['MLFLOW_TRACKING_INSECURE_TLS'] = 'true'
mlflow.set_tracking_uri(f"{MLFLOW_TRACKING_URI}")

# Get the experiment by name
experiment = mlflow.set_experiment(f"bike_sharing_{PARTICIPANT_FIRSTNAME}")

# Load all runs from the experiment
client = mlflow.tracking.MlflowClient()
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

# Display runs in a DataFrame
import pandas as pd

df_runs = pd.DataFrame([{
    "Run ID": run.info.run_id,
    "Run Name": run.data.tags.get("mlflow.runName"),
    "RMSE": run.data.metrics.get("rmse"),
    "R2": run.data.metrics.get("r2"),
    "Date": run.info.start_time
} for run in runs])

df_runs.sort_values("Date", ascending=False).reset_index(drop=True)

## 🏆 Select and Register the Best Model Run
The user is prompted to input the run name corresponding to the best-performing experiment. Based on this input:

- We locate the matching run and retrieve its unique run ID.

- Using the run ID, we construct the model URI to register the model in the MLflow Model Registry.

- The model is registered under a descriptive name (BikeSharingModel_{n_estimators}), enabling version control and easy deployment.

This process ensures the chosen model is formally tracked and available for production use.

In [ ]:
# Ask user to select a run name
selected_name = input("Enter the run name with the best performance to register its model: ")

# Find the corresponding run ID
selected_run = next(run for run in runs if run.data.tags.get("mlflow.runName") == selected_name)
run_id = selected_run.info.run_id

# Register the model from the selected run
model_uri = f"runs:/{run_id}/model"
MODEL_NAME = f"BikeSharingModel_{PARTICIPANT_FIRSTNAME}"
result = mlflow.register_model(model_uri, MODEL_NAME) 

print(f"Model registered: {result.name} v{result.version}")